In [1]:
# Import libaries and cleaned data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import requests
import plotly.express as px
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from matplotlib.pyplot import scatter
from IPython.display import HTML


merged = pd.read_csv("data/processed/merged_dataset.csv")

# Banking on the Future: Who's Still Left Out of the Global Financial System?

*In 2025, more than a billion adults remain unbanked. The data tells us a story about geography, gender, education, and a phone-shaped exit ramp.*

---

Imagine trying to live your life without a bank account. No direct deposit for your wages, no debit card, no safe way to save money for emergencies or take loans for housing and education. For more than just a billion adults globally, this isn't just a thought experiment. It's their reality. 

The World Bank's [Global Findex Database](https://www.worldbank.org/en/publication/globalfindex) has tracked financial inclusion for over a decade, and it's 2025 release shows a sharp increase in account ownership since 2011. But the progress hides a more worrying reality that the "unbanked" are not randomly scattered. They are grouped by country, by income, by gender, by schooling. Having a bank account in 2025 still depends, to an unsettling degree, on where you were born and who you happened to be born as.

This blog uses survey data from 152 countries paired with indicators from the World Bank API to reveal who is still locked out of the global financial system and what could be changing the picture.

## Data


*The dataset was built by combining two source families, both from the World Bank, joined country by country.*

---

**The Global Findex Database 2025** is a household-level survey covering more than 150 countries, run every three years. The survey asks individuals whether they own a financial account, how they use it, and how they manage their money more broadly. I downloaded the full 2025 release as a CSV which contains all data since 2011, and filtered it to the most recent survey year, 2024. 
The raw data contains multiple rows for country, one for each demographic, so I only kept the first row from each country, the all-adults total. The indicators I kept were the ones relevant for my analysis. I renamed each column to something more readable than its original survey code.

**The World Bank Open Data API** was useful for pulling the breakdowns of demographics and GDP per capita data in the flat format I wanted. I pulled both using the requests library and a small reusable function that hits the relevant indicator endpoint and returns a tidy two-column DataFrame. The demographics I wanted to look at were income level, gender, and education. The API returns regional aggregates aswell as countries which I had to filter out using a seperate metadata call so the merged file contains only sovereign countries

The sources shared the same ISO-3 country code so could be merged cleanly on a single key. The Findex region field bundles geography eith income group (e.g "Europe & Central Asia (excluding high income)"), which will make grouping and plotting difficult later on. So I made one more API call for metadta and attached a clean geographic region column. The resulting merged dataset contains data from 152 countries with 17 variables. Every chart and regression in this post is built from that single file.

## The view from above

Starting with a map, we can get a good overview of the percentage of adults who report owning an account at a bank, credit union, microfinance institution, or via mobile money in each country. Hover over a country to see the share.

In [2]:
# Only keep necessary columns for analysis
world_map_analysis = merged[['country', 'country_code', 'percent_with_fin_account']].copy()

#Rewrite percentages as percentages instead of decimals
world_map_analysis['percent_with_fin_account'] = world_map_analysis['percent_with_fin_account'] * 100

# Create the first choropleth map showing percent with financial account by country
world_map_fig = px.choropleth(
    world_map_analysis,
    locations='country_code',
    color='percent_with_fin_account',
    hover_name='country',
    color_continuous_scale="Blues",
    range_color=(0, 100),
    projection="robinson",
    labels={'percent_with_fin_account': 'Account Ownership (%)'},
    title='Percentage of Adults with a Financial Account by Country'
)

world_map_fig.update_geos(showframe=False, showcoastlines=False)
world_map_fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
world_map_fig.show()


*Figure 1: Share of adults (15+) with a financial account, latest year available. Source: World Bank Global Findex Database*

There is a clear global divide in account ownsership. North America, Western Europe and Australia are all dark blue signifying most of the adult population owning an account. Most of South America and Asia sits in the middle. Sub-Saharan Africa, parts of North Africa and pockets of the Middle East stay light. Countries with the lowest account ownership have rates well below 30%, meaning majority of their adult population manage their financial lives entirely outside the formal financial system.

The map's main pattern is unmistakable: financial inclusion is heavily associated with economic development, with richer countires benefiting from stronger financial infrastructure and institutions. Richer regions tend to have much higher account ownership rates (80%-100%), while poorer countries are significantly behind (below 50%). 

However, there are notable exceptions. For example, Kenya in East Africa exhibits relatively high levels of account ownership despite its lower level of economic development. This is likely due to the widespread adoption of the mobile money technology, M-Pesa which has expanded rapidly since its introduction in 2007. This indicates that innovation and policy can play a crucial tole in expanding access, a theme explored further in the analysis below.




---


## Where the gap is biggest

To look at the regional pattern, we now take a look at that same data collapsed into World Bank's geographic groupings. The graph shows the average share of adults with a account ownership in that region.

In [3]:
# Define function to create a weighted average
def weighted_avg(df, value_col, weight_col="adult_pop"):
    sub = df.dropna(subset=[value_col, weight_col])
    return (sub[value_col] * sub[weight_col]).sum() / sub[weight_col].sum()

#Calculate average percent with financial account by geographic region
region_df = (
    merged.groupby("region_geo")
    .apply(lambda g: weighted_avg(g, "percent_with_fin_account"),
           include_groups=False)
    .reset_index(name="percent_with_fin_account")
    .sort_values("percent_with_fin_account", ascending=True)
    )

#Create bar chart showing average percent with financial account by geographic region
region_fig = px.bar(
    region_df,
    x="percent_with_fin_account",
    y="region_geo",
    orientation='h',
    text=region_df["percent_with_fin_account"].mul(100).round(1).astype(str) + '%',
    color="percent_with_fin_account",
    color_continuous_scale="Blues",
    title='Financial Inclusion by Geographic Region (avg % of adults with a financial account)',
    labels={'percent_with_fin_account': 'Average Account Ownership (%)', 'region_geo': ""}
)

region_fig.update_traces(textposition='outside')
region_fig.update_layout(coloraxis_showscale=False, xaxis_tickformat=".0%")
region_fig.show()

*Figure 2: Average share of adults (15+) with a financial account by World Bank region*

As expected, North America and Europe & Central Asia sit top of the chart with North America at near-saturation, almost every adult has a financial account. At the other end, Sub-Saharan Africa and the World Bank's "Middle East, North Africa, Afghanistan and Pakistan" grouping fall well behind. The gap between the top and bottom regions is more than 50 percentage points. This is a chasm, not a gradient.

The chart starts to push back at the argument of wealth explaining inclusion that the map seemed to tell. South Asia comes in at 84%, while Latin America and Caribbean have sits at 70%. On a GDP-per-capita measure, the ordering doesn't make sense as on average, South Asians are considerable poorer than Latin Americans.

South Asia's high penetration is largely due to the sheer weight of India, which has run one of the most aggresive financial-inclusion drives of the last decade. The Jan Dhan Yojana scheme launched in 2014, paired with the Aadhaar digital identity scheme, opened hundreds of millions of new accounts. 

Population weighting also explain why Sub-Saharan Africa sits where it does, second to last at 58%. Mobile money services in Kenya, Tanzania, Ghana and Nigeria have put hundreds of millions of adults onto the financial system. MENA + Afhghanistan + Pakistan illustrates the opposite mechanism. The region's largest populations (Pakistan, Egypt, Iran) all have lower inclusion than the region's average country, so weighting pulls the number down.



## The unbanked are not a random sample

Within countries, the people who don't own financial accounts are not random. Using demographics taken from the World Bank API, and pooling these across countries yields the graph below.

In [4]:
# Create a list of demographic account ownership columns to convert from percentages to decimals
demo_cols = ["account_female", "account_male", "account_poor40", 
             "account_rich60", "account_prim_less", "account_sec_more"]

# Divide by 100 to convert to decimals to match the percent_with_fin_account variable
for col in demo_cols:
    merged[col] = merged[col] / 100     

# Compute population-weighted average percent with financial account for each demographic group

def weighted_unbanked(df, account_col, weight_col="adult_pop"):
    """Population-weighted unbanked rate (1 - account ownership)."""
    sub = df.dropna(subset=[account_col, weight_col])
    weighted_account = (sub[account_col] * sub[weight_col]).sum() / sub[weight_col].sum()
    return 1 - weighted_account

rows = [
    ("Gender",    "Women",              weighted_unbanked(merged, "account_female")),
    ("Gender",    "Men",                weighted_unbanked(merged, "account_male")),
    ("Income",    "Poorest 40%",        weighted_unbanked(merged, "account_poor40")),
    ("Income",    "Richest 60%",        weighted_unbanked(merged, "account_rich60")),
    ("Education", "Primary or less",    weighted_unbanked(merged, "account_prim_less")),
    ("Education", "Secondary or more",  weighted_unbanked(merged, "account_sec_more")),
]

unbanked_df = pd.DataFrame(rows, columns=["dimension", "group", "unbanked_rate"])

# Create bar chart showing unbanked rates by demographic group
unbanked_df["Status"] = unbanked_df["group"].map({
    "Women": "Disadvantaged group",
    "Men": "Advantaged group",
    "Poorest 40%": "Disadvantaged group",
    "Richest 60%": "Advantaged group",
    "Primary or less": "Disadvantaged group",
    "Secondary or more": "Advantaged group",
})

unbanked_fig = px.bar(
    unbanked_df,
    x="dimension",
    y="unbanked_rate",
    color="Status",
    barmode="group",
    hover_data=["group"],
    text=unbanked_df["unbanked_rate"].mul(100).round(1).astype(str) + "%",
    color_discrete_map={"Advantaged group": "#cccccc", "Disadvantaged group": "#1f77b4"},
    title="Unbanked rates by gender, income, and education",
    labels={"unbanked_rate": "% unbanked", "dimension": ""},
)
unbanked_fig.update_traces(textposition="outside")
unbanked_fig.update_layout(yaxis_tickformat=".0%", height=500, plot_bgcolor="white")
unbanked_fig.show()



*Figure 3: Population-weighted unbanked rates by demographic group.*

The graph illustrates three gaps. The first, the gender gap, is real but modest: about 4 percentage points more women than men are unbanked. The income gap is larger with 11% seperating the poorest 40 percent of adults from the richest 60 percent in their respective countries. The widest of the three is the education gap. Adults with primary education or less are unbanked roughly twice the rate of those with secodnary education or more.

The data tells us that people who never got past primary school are sytematically harder to reach with formal financial services, in rich and poor countries alike. However, correlation is not causation. People with primary education or less are also disproportionately concentrated in countries with weaker infrastrucure and less-developed financial systems. So some of what looks like "less education means more unbanked" is really "less-educated people happned to live in places where everyone is more unbanked". The same caveat applies, more mildly,to the gender and income gaps. Account ownership tracks the economic development and financial infrastructure of the place you happen to live, more than almost anything else about you.

## How much does money follow money?

The most intuitive driver of inclusion is wealth. Richer countries can build banking infrastructure and their citizens have more wealth that they'd like to deposit. The chart below plots every country's GDP per capita against its account ownership rate. Each bubble is a country, sized by its adult populaition and colored by region.

In [5]:
# Drop rows missing either variable for the scatter

scatter_data = merged.dropna(subset=["gdp_per_capita", "percent_with_fin_account"])

scatter.fig = px.scatter(
    scatter_data,
    x="gdp_per_capita",
    y="percent_with_fin_account",
    color="region_geo",                   # color by region tells the story
    size="adult_pop",                     # bubble size = adult population
    size_max=60,
    hover_name="country",
    hover_data={
        "gdp_per_capita": ":$,.0f",
        "percent_with_fin_account": ":.0%",
        "adult_pop": ":,.0f",
        "region_geo": False,              # hide since it's already the color
    },
    log_x=True,                            # critical — GDP spans 3 orders of magnitude
    title="Wealthier countries are more financially included",
    labels={
        "gdp_per_capita": "GDP per capita (log scale, USD)",
        "percent_with_fin_account": "Account ownership",
        "region_geo": "Region",
    },
    trendline="ols",                       # log-linear fit
    trendline_scope="overall",             # one fit across all countries, not per region
    trendline_color_override="black",
)

scatter.fig.update_traces(marker=dict(line=dict(width=0.5, color="white")))
scatter.fig.update_layout(
    yaxis_tickformat=".0%",
    height=600,
    plot_bgcolor="white",
    legend=dict(title="", yanchor="bottom", y=0.02, xanchor="right", x=0.98),
)
scatter.fig.show()

*Figure 4: GDP per capita (log scale) versus account ownership. Bubble size is adult population and the black line is a log-linear best fit.*

The relationship is unmistakable. As GDP per capita rises so does account ownership. As shown economic development is a key driver of financial inclusion.

The non-linear pattern shows a curve that is flat at low income levels and increases more steeply as income rises. At low levels of income, increases in GDP are associated with relatively small improvements in financial inclusions suggesting the presence of structural barriers. As economies develop, financial systems scale up more rapidly. 

Importantly, there remians substantial variation across countries at similiar income levels, highlighting factors beyond income such as institutions, regulation, and financial innovation also play a significant role. Fitting a simple log-linear regression of account ownsership on log GDP per capita results in an R-squared value of 0.59. To explain this clearly, the single variable of how rich a country is explains nearly 60 percent of the variation in financial inclusion across the world. This is a large percentage but it also means 40 percent of the story is somethinbg else. 

## Mobile money: the unexpected leapfrog

The Findex tracks two ways of having a financial account: a traditional bank or financial institution account, and a mobile money account which is an electronic wallet service, usually operated by a telecom provider rather than a bank. The chart below shows the regional account ownserhsip percentage of the two channels.

In [8]:
# Population-weighted averages by region for both account types

regional = (
    merged.groupby("region_geo")
    .apply(lambda g: pd.Series({
        "Traditional banking": weighted_avg(g, "percent_with_fin_account_formal"),
        "Mobile money":        weighted_avg(g, "percent_with_mobile_account"),
    }), include_groups=False)
    .reset_index()
    .melt(id_vars="region_geo", var_name="Account type", value_name="share")
)

# Sort regions by traditional banking ascending, so the contrast pops visually
region_order = (
    regional[regional["Account type"] == "Traditional banking"]
    .sort_values("share")["region_geo"].tolist()
)

mobile_fig = px.bar(
    regional,
    x="share",
    y="region_geo",
    color="Account type",
    barmode="group",
    orientation="h",
    category_orders={"region_geo": region_order},
    color_discrete_map={
        "Traditional banking": "Blue",
        "Mobile money": "Orange",
    }, 
    labels={"share": "Share of adults", "region_geo": "", "Account type": ""},
)
mobile_fig.update_traces(textposition="outside")
mobile_fig.update_layout(
    xaxis_tickformat=".0%",
    height=500,
    plot_bgcolor="white",
)
mobile_fig.show()


/var/folders/rn/1ww2hhjn0r58pz6lrgp_90sw0000gn/T/ipykernel_2964/957476742.py:4: RuntimeWarning:

invalid value encountered in scalar divide



*Figure 5: Population-weighted share of adults using traditional bank accounts vs. mobile money accounts, by region.*

The clear income gradient in traditional banking reinforces the argument we've been building: richer countries having higher account ownership rates due to their deeper financial infrastructure. Mobile money partly fills the gap. The regions with the lowest share of adults using traditional bank accounts such as, Sub-Saharan Africa and Latin America are also the regions with the highest share using mobile money. 

In Sub-Saharan Africa, mobile money adoption is relatively high depite the low share of adults with a traditional bank account and arguably because of it. Services like M-Pesa, MTN MoMo and EcoCash have reached hundreds of millions of adults that traditional banks never did, stepping into the gap that weak financial infrastructure left open.

The key takeaway is that rich countries are still riding bank-led inclusion leaning on decades-old institutional plumbing. Whereas the poor and lower-middle-income countries are increasingly riding tech-led inclusion. The innovation of mobile money services is a relatively new one with M-Pesa launching in 2007 and the Findex not tracking mobile money as a seperate category until 2014. The rapid spread in the years since points to a clear lesson for other developing countries still building out thing banking networks.

## What actually predicts inclusion?

To see relationships between different variables and financial inclusions while controlling for other factors, I ran four versions of the same regression, each one adding a variable. Robust standard errors (HC3) are used to account for potential heteroskedacity in the data, ensuring the inference remains valid.

In [7]:

HTML(open("outputs/regression_table.html").read())

*Table 1: OLS regressions of account ownership on country-level predictors. Coefficients on log GDP per capita can be read as: a one-log-point increase in GDP per capita (roughly a 2.7x increase) is associated with this many additional percentage points of account ownership*

The table gives us a clear image of what predicts financial inclusion. Log GDP per capita is the strongest baseline predictor, it's significant at the 1% level in every specification. With a coefficient ranging from 0.11 to 0.14 meaning a one-log-point increase in GDP per capita is associated with roughly 11 14 additional percentage points of account ownership. As discussed throughough the blog, wealth matters.

Another interesting insight revealed from the table is that accounting for the region rixed effects increased the R-squared value from 0.595 to 0.700. So controlling for regional heterogeneity significantly improves explanatory power, suggesting structural differences across regions.

The most striking result is the large, and statistically significant, coefficient of mobile money adoption. A 10 percentage point increase in mobile money adoption within a country is associated with roughly a 5 percentage point increase in overall account ownership. Mobile money isn't just substituting for a bank account that would have been opened anyway, it's reaching adults who would otherwise be unbanked. (Note: that specificiation drops to 79 countries because mobile oney data isn't reported everywhere.)


## Conclusion

As discussed throughougt the entirety of this blog, the single biggest correlae of inclusion is national income. That's the unsuprising story and it's largely a story about decades of economic development. However, we have seen the success of a new channel in the most recent decades that has had huge success in countries like. Mobile money is the clearesy example we have that a country doesn't have to wait for a banking system to catch up to the rest of it's economy.

And then there's the demographic story. The gap between people with secondary education and people without it, the gap between women and men, the gap between the richest 60 percent of the country versus the poorest 40 percent. These gaps won't be closed by smarter apps. Finanical literacy programs, basic numeracy curricula and outreach to people are some of the important steps that need to be taken.

The world has narrowed the financial inclusion gap dramatically over the last couple of decades, through state-led account drives in India and mobile money in Sub-Saharan Africa. But there's still a long way to go. The case for financial inclusion is a strong one. A formal account is what lets a family save for an emergency, take a loan out, receive government payments. A 2016 study of M-Pesa alone estimated it lifted 200,000 Kenyan households out of extreme poverty. Financial inclusion isn't just an indicator of development. It's a quiet engine of it.

___